 ## Human In The Loop
案例：在调用工具前，需要人工确认 

In [48]:
from distutils.command.config import config

from dotenv import load_dotenv
import os
load_dotenv()

True

In [49]:
from langchain.tools import tool, ToolRuntime
## 创建两个读取、发送邮件的虚拟工具
@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from state."""
    # take email from state 
    return runtime.state['email']

@tool
def send_email(runtime: ToolRuntime,message) -> str:
    """Send an email message to the email address in state."""
    # fake email sending 
    return f"Email sent：{message}"

In [50]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_deepseek import ChatDeepSeek

class EmailState(AgentState):
    email: str

llm_ds = ChatDeepSeek(model='deepseek-chat',
                 api_key=os.getenv('DEEPSEEK_API_KEY'),
                      )
    
agent = create_agent(
    model=llm_ds,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            'read_email': False,
            'send_email': True,
        },
        description_prefix='工具使用需要人工确认。'
    )]
)
    

In [54]:
from langchain.messages import HumanMessage, AIMessage

config = {'configurable':{'thread_id': '2'}}

responses = agent.invoke({
    'messages':HumanMessage(content="请读取我的邮件地址，并发送一条消息（调用工具完成这一任务）"),
    'email': 'Hi there, this is a test email.'
}, config=config)


In [55]:
from pprint import pprint
pprint(responses)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'message': '你好！这是一条测试消息。祝你工作顺利，生活愉快！😊'},
                                                         'description': '工具使用需要人工确认。\n'
                                                                        '\n'
                                                                        'Tool: '
                                                                        'send_email\n'
                                                                        'Args: '
                                                                        "{'message': "
                                                                        "'你好！这是一条测试消息。祝你工作顺利，生活愉快！😊'}",
                                                         'name': 'send_email'}],
                                    'review_configs': [{'action_name': 'send_email',
                                                        'allowed_decisions': ['approve',
                                 

In [57]:
pprint(responses['__interrupt__'])


[Interrupt(value={'action_requests': [{'args': {'message': '你好！这是一条测试消息。祝你工作顺利，生活愉快！😊'},
                                       'description': '工具使用需要人工确认。\n'
                                                      '\n'
                                                      'Tool: send_email\n'
                                                      "Args: {'message': "
                                                      "'你好！这是一条测试消息。祝你工作顺利，生活愉快！😊'}",
                                       'name': 'send_email'}],
                  'review_configs': [{'action_name': 'send_email',
                                      'allowed_decisions': ['approve',
                                                            'edit',
                                                            'reject']}]},
           id='e9e7a5b810a9aa4e72a623e635ce4fb8')]


In [58]:
from langgraph.types import  Command

res = agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ),
    config=config, # Same thread ID to resume the paused conversation
    version="v2",
)

In [59]:
print(res)

GraphOutput(value={'messages': [HumanMessage(content='请读取我的邮件地址，并发送一条消息（调用工具完成这一任务）', additional_kwargs={}, response_metadata={}, id='377a1be5-6723-41ef-a09a-977b9d050de0'), AIMessage(content='好的，我先读取当前状态中的邮件地址。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 319, 'total_tokens': 356, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 63}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'id': '38ce16a2-0216-4ebb-9ee2-9844dec668f5', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e15fd-f74b-7e91-a722-923ae8bb5cff-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': 'call_00_jkQt88WkFSRF56vAXgMC0133', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 319, 'output_tokens': 37,